In [99]:
!pip -q install -U "mcp>=2,<3" google-genai

# Pydantic Data Modeling: User Profile, Skills, Work experiences and Achievements
##### Building Structured, Validated, and Reusable Python Models Using Pydantic v2

In [100]:

from pydantic import BaseModel, Field, ConfigDict

class BaseSchema(BaseModel):
  model_config = ConfigDict(
      populate_by_name=True,
      extra="ignore",
      str_strip_whitespace=True,
  )

class Profile(BaseSchema):
  name: str
  email: str
  role: str
  phone: str
  location: str
  image: str
  linkedin: str
  github: str
  about: str


class Skill(BaseSchema):
  name: str
  achieved: str
  title: str
  icon: str
  is_hidden: bool = Field(
      default=False,
      alias="isHidden",
  )

class Company(BaseSchema):
  name: str

class WorkExperience(BaseSchema):
  company: Company
  position: str
  job_type: str = Field(
      alias= "jobType"
  )
  duration: str
  location: str
  responsibilities: list[str]

class User(BaseSchema):
  profile: Profile
  achievements: list[str] = Field(default_factory=list)
  skills: list[Skill] = Field(default_factory=list)
  experiences : list[WorkExperience] = Field(
      alias="workExperiences"
  )


# Asynchronous API Data Fetching with HTTPX and Pydantic
#### Fetching JSON Data, Handling HTTP Errors, and Validating API Responses

In [101]:
import asyncio
import httpx

async def get_user():
    async with httpx.AsyncClient() as client:
        response = await client.get(
            "https://raw.githubusercontent.com/arifuzzaman-tanin/test-json-public-db/refs/heads/main/database.json"
        )

        response.raise_for_status()

        return  User.model_validate(response.json())

# Building an MCP Server for User Profile Data
#### Exposing an Asynchronous MCP Tool to Retrieve Structured User Information

In [102]:
from datetime import date
from dateutil.relativedelta import relativedelta
from mcp.server import MCPServer

mcp = MCPServer("User Server")

@mcp.tool()
async def fetch_my_profile() -> User:
  """Retrieve my profile information. User Profile, Skills, Work experiences and Achievements"""

  return await get_user()

In [110]:
@mcp.tool()
async def get_total_users() -> int:
    """Retrieve the total number of registered users."""
    return 100

# Initializing Google Gemini AI Client
##### Configuring the Gemini API Using Google GenAI SDK and Secure Google Colab Secrets

In [103]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

gemini = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini initialized!")

Gemini initialized!


# Building an AI-Powered MCP Client with Google Gemini
##### Dynamic Tool Discovery, Intelligent Tool Selection, MCP Execution, and AI-Generated Responses

In [104]:

import json
from google.genai import types
from mcp import Client


async def ask_ai(question: str) -> str:
    """Answer questions using data retrieved from MCP tools."""

    async with Client(mcp) as client:

        # Discover available MCP tools
        tools_result = await client.list_tools()

        available_tools = [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema,
            }
            for tool in tools_result.tools
        ]

        # Ask Gemini to select an MCP tool
        decision = await gemini.aio.models.generate_content(
            model=MODEL,
            contents=f"""
            You are an AI assistant for a personal portfolio.

            Available MCP tools:
            {json.dumps(available_tools)}

            User question:
            {question}

            Select the most appropriate MCP tool.

            Return JSON containing:
            - tool: selected tool name or null
            - arguments: tool arguments as an object

            Only select tools from the available list.
            """,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            ),
        )

        selected = json.loads(decision.text)

        tool_name = selected.get("tool")
        arguments = selected.get("arguments", {})

        # Validate selected tool
        allowed_tools = {
            tool.name for tool in tools_result.tools
        }

        if tool_name is None:
            return "I don't have enough information to answer that."

        if tool_name not in allowed_tools:
            return "The selected MCP tool is unavailable."

        if not isinstance(arguments, dict):
            return "Invalid MCP tool arguments."

        # Execute the MCP tool
        result = await client.call_tool(
            tool_name,
            arguments,
        )

        tool_output = "\n".join(
            item.text
            for item in result.content
            if item.type == "text"
        )

        if result.is_error:
            return f"MCP tool error: {tool_output}"


        # Generate a beautifully formatted answer
        response = await gemini.aio.models.generate_content(
            model=MODEL,
            contents=f"""
            You are a professional AI portfolio assistant.

            User question:
            {question}

            Retrieved portfolio data:
            {tool_output}

            Instructions:
            - Answer using only the retrieved portfolio data.
            - Use well-structured Markdown formatting.
            - Use # for the main heading.
            - Use ## for section headings.
            - Use **bold** for important information.
            - Use bullet points for lists.
            - Use Markdown tables when comparing information.
            - Keep the answer clear, professional, and concise.
            - Do not invent information.
            - Do not wrap the answer in a Markdown code block.
            """,
        )

        return response.text or "No response generated."


# Ask Questions

In [105]:
from IPython.display import display, Markdown

async def main():
    answer = await ask_ai(
        "Based on everything you know about me, challenge three assumptions I might have about my own career."
    )

    display(Markdown(answer))


await main()

# Challenging Your Career Assumptions

As a **Senior Full-Stack Software Engineer** with over 9 years of experience spanning traditional enterprise stacks, cloud architecture, and modern Generative AI, your career profile presents some distinct patterns. 

Here are three assumptions you might be holding about your career trajectory, challenged using your professional data:

## 1. Assumption: "I am primarily a traditional web developer who just dabbles in AI."
* **The Reality:** While your foundation is firmly rooted in C#, .NET Core, React, and SQL databases, your recent role at **ZONDA** and your continuous upskilling (such as the upcoming *Gen AI & ML Professional Program* and hands-on work with **LangChain, LangGraph, ChromaDB, vector retrieval, and AI agents**) indicate that you are actively operating at the intersection of full-stack engineering and Applied AI/MLOps. 
* **The Challenge:** You are no longer just a traditional full-stack developer; you are rapidly transitioning into an **AI-augmented systems engineer** who builds end-to-end intelligent applications. You should position yourself not just as a .NET/React developer, but as a specialist capable of bridging heavy backend architecture with modern LLM pipelines.

## 2. Assumption: "My value lies entirely in writing application code."
* **The Reality:** Your experience goes far beyond writing features. Across multiple roles—from optimizing Elasticsearch schemas at ZONDA (resulting in a **30% search-performance improvement**) and reducing release cycles by **40% at PETRONAS**, to tuning SQL queries that improved healthcare-system performance by **30% at SIM Solution Limited**—your real superpower is **system optimization and reliability**.
* **The Challenge:** Your career growth isn't just about mastering new programming languages; it is about architectural impact. Your ability to handle CI/CD, Terraform, Docker, Kubernetes, and observability tools like Datadog and Application Insights means your core value is delivering **scalable, high-performance production ecosystems**, not just lines of code.

## 3. Assumption: "My background is purely technical and isolated from business outcomes."
* **The Reality:** Your work history highlights continuous collaboration with stakeholders, product teams, and QA across Agile/Scrum environments. You have built fintech, SaaS, e-commerce, real estate, and healthcare systems where business continuity, secure data exchange, and client-facing delivery are critical.
* **The Challenge:** You often view your work through a purely engineering lens (e.g., APIs, microservices, databases), but your technical contributions directly drive business metrics—such as reducing security incidents by 40% or speeding up data retrieval. You should lean heavier into your **technical leadership and cross-functional collaboration** capabilities when looking toward your next career milestone.

In [106]:
from IPython.display import display, Markdown

async def main():
    answer = await ask_ai(
        "If I applied for a CTO position tomorrow, what would be my biggest weaknesses?"
    )

    display(Markdown(answer))


await main()

# Evaluation of CTO Readiness: Potential Weaknesses

Based on Arifuzzaman Tanin's current portfolio data, stepping directly into a Chief Technology Officer (CTO) position tomorrow would present several notable gaps. While Arifuzzaman possesses strong technical depth as a Senior Full-Stack Developer with 9+ years of experience in modern web technologies, cloud platforms, and emerging Generative AI, a CTO role demands a broader shift from hands-on engineering execution to executive leadership, organizational scaling, and business strategy.

## Key Potential Weaknesses for a CTO Position

* **Lack of Explicit Executive & C-Suite Experience:** The portfolio demonstrates strong individual contributor and technical delivery roles (e.g., Senior Full-Stack Developer, Software Engineer), but does not explicitly show experience operating at the executive level (such as managing an entire engineering organization, reporting directly to a board of directors, or owning enterprise-wide technology strategy).
* **Limited Financial & Budgetary Governance:** A core responsibility of a CTO is managing P&L (Profit and Loss), setting departmental budgets, forecasting cloud infrastructure costs at scale, and calculating ROI on technology investments. The provided data focuses purely on technical delivery rather than financial management.
* **Organizational Scale & Enterprise Governance:** While Arifuzzaman has experience mentoring developers and leading technical discussions, the portfolio lacks explicit evidence of managing multi-layered engineering teams, establishing company-wide hiring strategies, or scaling organizational culture across multiple international departments.
* **Macro-Level Technology Vision vs. Hands-on Execution:** The resume highlights deep, hands-on involvement in writing code, building RAG pipelines, optimizing Elasticsearch, and deploying containers. A CTO must transition from direct implementation to high-level architectural oversight, vendor management, and long-term strategic roadmapping.

## Summary of Profile Alignment

| Leadership Domain | Current Portfolio Strength | Potential CTO Gap |
| :--- | :--- | :--- |
| **Technical Architecture** | Strong (.NET, Cloud, Microservices, GenAI, MLOps) | Enterprise-wide multi-cloud/hybrid governance strategy |
| **Team Leadership** | Mentoring, code reviews, cross-functional collaboration | Managing engineering managers, org design, and department scaling |
| **Business Strategy** | Client-facing delivery and requirement translation | Corporate budgeting, P&L ownership, and board-level communication |

## Conclusion
While Arifuzzaman Tanin is a highly capable **Senior Full-Stack Developer** with advanced skills in cloud-native systems and modern AI pipelines, applying for a CTO position tomorrow would highlight a typical transition gap: moving from deep technical execution and team-level mentorship to executive leadership, financial governance, and enterprise-wide technology strategy.

In [107]:
from IPython.display import display, Markdown

async def main():
    answer = await ask_ai(
        "Based on my experience, which certification would provide the highest career value with the least preparation time?"
    )

    display(Markdown(answer))


await main()

# Certification Recommendation

Based on your extensive 9+ years of professional experience as a **Senior Full-Stack Developer** with deep, hands-on expertise in **Azure**, **AWS**, and **.NET**, the following certifications will provide the highest career value with the least preparation time, as they align directly with your daily technical stack and work history.

## Top Certification Recommendations

| Certification Path | Target Platform / Technology | Relevance to Your Experience | Preparation Effort |
| :--- | :--- | :--- | :--- |
| **Microsoft Certified: Azure Developer Associate (AZ-204)** | Azure (.NET, Functions, App Services, Service Bus, AKS) | **Very High**: Matches your enterprise work at Petronas and Zonda. | **Low to Moderate** (Leverages your day-to-day Azure experience) |
| **AWS Certified Solutions Architect – Associate (SAA-C03)** | AWS (Lambda, EKS, DynamoDB, S3, RDS) | **Very High**: Matches your current architecture work at Zonda. | **Low to Moderate** (Builds directly on your current AWS stack) |
| **Microsoft Certified: Azure AI Engineer Associate (AI-102)** | Generative AI, Azure AI Foundry, OpenAI | **High**: Complements your current focus on RAG, AI agents, and LLM pipelines. | **Moderate** (Aligns with your recent Generative AI explorations) |

---

## Strategic Recommendation

* **Primary Recommendation:** **Microsoft Certified: Azure Developer Associate (AZ-204)** or **AWS Certified Solutions Architect – Associate (SAA-C03)**. 
* **Why:** Because you already design, deploy, and maintain cloud-native systems, serverless infrastructure, and microservices on both **Azure** and **AWS**, you will require the least preparation time while instantly validating your senior-level cloud competencies. 
* **Emerging Edge:** If you want to lean further into your growing portfolio focus on **Generative AI**, targeting **Azure AI Engineer Associate** will strongly validate your applied skills in LLMs, LangChain, and RAG architectures.

In [108]:
from IPython.display import display, Markdown

async def main():
    answer = await ask_ai(
        "What is my biggest professional weakness that I might not recognize myself?"
    )

    display(Markdown(answer))


await main()

# Professional Self-Reflection: Potential Hidden Weakness

Based on the provided portfolio data, Arifuzzaman Tanin is a **Senior Full-Stack Software Engineer** with **9+ years of experience**, extensive technical breadth spanning traditional backend/frontend stacks (.NET, React, Python, AWS, Azure), and a rapid expansion into cutting-edge domains (Generative AI, RAG, AI agents, and MLOps). 

While his resume showcases strong technical delivery, optimization capabilities, and cross-functional collaboration, a professional in this phase of a career often faces a specific blind spot that might not be immediately recognized.

## The Potential Hidden Weakness

* **Over-Diversification and Context Switching:** 
  * **The Challenge:** Arifuzzaman's skill set and recent project responsibilities are exceptionally broad. He actively works across full-stack web development (.NET Core, React, Next.js), cloud infrastructure (AWS, Azure, Terraform, Docker, Kubernetes), databases (PostgreSQL, Elasticsearch, Redis), and advanced emerging technologies (Generative AI, LLM pipelines, RAG, LangChain, LangGraph, Python machine learning, and MLOps). 
  * **Why It Might Not Be Recognized:** Because he successfully delivers across all these domains, it feels like high productivity and versatility. However, maintaining a world-class level of depth across *both* heavy enterprise cloud/backend systems *and* fast-moving Generative AI/MLOps engineering can lead to cognitive fatigue or "jack-of-all-trades" friction. It can obscure the point where high competence in many areas begins to crowd out absolute mastery in a single domain.

## Summary of Professional Profile Strengths vs. Hidden Risks

| Area | Observed Strengths | Potential Blind Spot / Hidden Risk |
| :--- | :--- | :--- |
| **Technical Range** | Mastery of .NET, React, Cloud (AWS/Azure), and modern GenAI (LangChain, vector databases). | Spreading focus too thin across rapidly evolving ecosystems, risking superficial depth in hyper-specialized sub-fields. |
| **Execution & Delivery** | Proven track record of performance optimization (e.g., 30% search improvement, 40% release cycle reduction). | Constantly taking on multi-layered roles (full-stack + cloud + AI/ML) which may lead to hidden burnout or bottlenecks in team dependency. |
| **Adaptability** | Seamlessly transitioning from traditional enterprise SaaS (Fintech, Healthcare) to modern AI agents and MLOps. | Balancing legacy architecture mental models with experimental, non-deterministic AI paradigms. |

*Note: This insight is drawn strictly from the breadth of multi-domain responsibilities listed in the portfolio data and represents a structural challenge common to high-performing senior engineers moving into emerging tech fields.*

In [109]:
from IPython.display import display, Markdown

async def main():
    answer = await ask_ai(
        "If I had to remove half of my technical skills from my resume, which ones should I keep and why?"
    )

    display(Markdown(answer))


await main()

# Technical Skills Strategy: What to Keep When Reducing Your Resume by Half

As a **Senior Full-Stack Developer** with extensive experience in cloud engineering, modern web development, and emerging AI/GenAI technologies, streamlining your technical skills requires focusing on core competencies that drive high impact. 

If you must reduce your technical skills by half, here is the strategic selection of the **core skills to keep**, chosen based on your professional background and current industry demands.

## Recommended Skills to Keep

| Category | Skills to Keep | Why Keep It? |
| :--- | :--- | :--- |
| **Backend & APIs** | **C#, .NET Core, ASP.NET Core Web API, REST APIs, Microservices** | Forms the backbone of your **9+ years of professional experience** building enterprise-grade backend services, scalable architectures, and REST APIs across multiple companies (ZONDA, Petronas, and past roles). |
| **Frontend** | **TypeScript, JavaScript, React, Next.js** | Essential for your full-stack capabilities, directly tied to your current work on production platforms like Livabl.com and NewHomeSource.com. |
| **Generative AI & ML** | **LLM pipelines, RAG, agentic AI, LangChain, LangGraph, Python** | Represents your **growing focus and cutting-edge work** in Generative AI, MLOps, vector search, and AI agents at ZONDA, keeping your profile future-ready. |
| **Cloud & DevOps** | **AWS, Azure, Docker, Kubernetes, CI/CD (GitHub Actions, Azure DevOps)** | Critical for your expertise in cloud-native deployment, containerization, and managing production environments across AWS and Azure infrastructure. |
| **Databases** | **PostgreSQL, SQL Server, Redis, Elasticsearch** | Essential relational, caching, and search technologies you actively tune and optimize (such as Elasticsearch schemas for a 30% search-performance boost). |

---

## Why These Specific Skills Were Chosen

* **Core Professional Identity:** Your primary title is **Senior Full-Stack Software Engineer** specializing in C#, .NET, Python, React, and cloud ecosystems. Dropping these would obscure your core engineering identity.
* **High-Impact Production Value:** The retained skills directly align with measurable achievements in your work experience, such as improving search performance, reducing release cycles, and implementing advanced RAG and AI-agent workflows.
* **Market Relevance:** Combining strong traditional full-stack skills (.NET, React, SQL, Cloud) with modern specializations (Generative AI, Python, Kubernetes) ensures you remain competitive for senior and specialized roles.

In [112]:
from IPython.display import display, Markdown

async def main():
    answer = await ask_ai(
        "How many users are currently registered in the system?"
    )

    display(Markdown(answer))


await main()

# System User Statistics

## Overview
This section provides the current registration metrics for the system based on the latest portfolio data.

* **Metric:** Total Registered Users
* **Count:** **100**

## Summary Table

| Category | Total Count |
| :--- | :--- |
| **Registered Users** | **100** |